<div class="blog-language-switch" role="group" aria-label="文章语言">
<a href="../../Deep-Learning/17-deep-reinforcement-learning-world-models.html" lang="en" hreflang="en">English</a>
<span aria-current="page">中文</span>
</div>

[返回深度学习总览](Deep-Learning.html)

## **深度强化学习与世界模型** {#deep-reinforcement-learning-world-models}

深度强化学习（deep RL）研究的数据会受到智能体自身决策的影响。监督学习器接收固定的有标签样本对；RL 智能体则选择动作、改变下一步将观察到的状态，而且可能要经过很长时间才能获得有用反馈。神经网络为策略、价值函数与环境模型提供了可扩展表示，但也会放大自举误差、分布偏移和优化不稳定性。

![动作会改变 RL 智能体未来用于学习的观测。](assets/dl17-agent-environment-loop.svg){fig-align="center" width="76%" fig-alt="闭环中，智能体向环境发送动作，并接收下一观测、奖励和终止信号。"}

*基于 Sutton 与 Barto 的 [Reinforcement Learning: An Introduction](https://mitpress.mit.edu/9780262039246/reinforcement-learning/) 中智能体—环境接口绘制的原创机制图。*

可执行实验主线采用官方 [Gymnasium Pendulum-v1](https://gymnasium.farama.org/environments/classic_control/pendulum/) 规范：观测为 $s_t=[\cos\theta_t,\sin\theta_t,\dot\theta_t]$，连续扭矩为 $a_t\in[-2,2]$，每个 episode 最多 200 步，奖励为

$$
r_t=-\left(\theta_t^2+0.1\dot\theta_t^2+0.001a_t^2\right).
$$

本地实现遵循文档中的动力学方程，因此即使未安装可选的 Gymnasium 依赖，每个示例也能离线运行。一组固定轨迹种子按完整 episode 划分为训练、验证和测试分区。离散控制方法使用同一动作区间中的五种扭矩，连续控制方法保留原始动作空间。这些短程 CPU 实验用于审查机制，而不是复现论文 benchmark 分数。

![Gymnasium 为 Pendulum-v1 定义的坐标系。](assets/dl17-pendulum-coordinate.png){fig-align="center" width="46%" fig-alt="Pendulum-v1 官方坐标图，展示角度 theta 与施加的扭矩。"}

*来源：[Gymnasium Pendulum-v1 文档](https://gymnasium.farama.org/environments/classic_control/pendulum/)，Farama Foundation。环境代码与文档源文件随 Gymnasium 项目按照该项目的 MIT 许可证发布。*

<details>
<summary><strong>PyTorch：建立共享环境与按 episode 划分的数据</strong></summary>

```python
import copy
import math
import random
from collections import deque

import numpy as np
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)


def seed_everything(seed=1717):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def angle_normalize(theta):
    return ((theta + math.pi) % (2 * math.pi)) - math.pi


class LocalPendulum:
    """Offline-compatible implementation of the documented Pendulum-v1 dynamics."""

    def __init__(self, horizon=120, g=10.0, mass=1.0, length=1.0, dt=0.05):
        self.horizon, self.g, self.mass, self.length, self.dt = horizon, g, mass, length, dt
        self.max_speed, self.max_torque = 8.0, 2.0

    def reset(self, seed):
        self.rng = np.random.default_rng(seed)
        self.theta = float(self.rng.uniform(-math.pi, math.pi))
        self.theta_dot = float(self.rng.uniform(-1.0, 1.0))
        self.steps = 0
        return self.observe()

    def observe(self):
        return np.array([math.cos(self.theta), math.sin(self.theta), self.theta_dot], dtype=np.float32)

    def step(self, action):
        torque = float(np.clip(np.asarray(action).reshape(-1)[0], -self.max_torque, self.max_torque))
        theta, theta_dot = self.theta, self.theta_dot
        cost = angle_normalize(theta) ** 2 + 0.1 * theta_dot ** 2 + 0.001 * torque ** 2
        acceleration = 3 * self.g / (2 * self.length) * math.sin(theta) + 3 * torque / (self.mass * self.length ** 2)
        self.theta_dot = float(np.clip(theta_dot + acceleration * self.dt, -8.0, 8.0))
        self.theta = float(theta + self.theta_dot * self.dt)
        self.steps += 1
        truncated = self.steps >= self.horizon
        return self.observe(), -float(cost), False, truncated


def scripted_action(observation, rng, random_probability=0.55):
    if rng.random() < random_probability:
        return float(rng.uniform(-2, 2))
    theta = math.atan2(float(observation[1]), float(observation[0]))
    return float(np.clip(-2.0 * theta - 0.45 * observation[2] + rng.normal(0, 0.25), -2, 2))


def collect_episode(seed, horizon=120):
    env, rng = LocalPendulum(horizon=horizon), np.random.default_rng(seed + 50_000)
    observation = env.reset(seed)
    records = []
    for _ in range(horizon):
        action = scripted_action(observation, rng)
        next_observation, reward, terminated, truncated = env.step(action)
        records.append((observation, action, reward, next_observation, terminated or truncated))
        observation = next_observation
    return records


seed_everything()
episode_seeds = np.arange(17_000, 17_090)
episodes = [collect_episode(int(seed)) for seed in episode_seeds]
train_episodes, val_episodes, test_episodes = episodes[:60], episodes[60:75], episodes[75:]


def flatten_episodes(selected):
    states = torch.tensor(np.stack([x[0] for ep in selected for x in ep]), dtype=torch.float32)
    actions = torch.tensor([[x[1]] for ep in selected for x in ep], dtype=torch.float32)
    rewards = torch.tensor([[x[2]] for ep in selected for x in ep], dtype=torch.float32)
    next_states = torch.tensor(np.stack([x[3] for ep in selected for x in ep]), dtype=torch.float32)
    dones = torch.tensor([[x[4]] for ep in selected for x in ep], dtype=torch.float32)
    return states, actions, rewards, next_states, dones


train_s, train_a, train_r, train_ns, train_done = flatten_episodes(train_episodes)
val_s, val_a, val_r, val_ns, val_done = flatten_episodes(val_episodes)
test_s, test_a, test_r, test_ns, test_done = flatten_episodes(test_episodes)

assert train_s.shape == (7200, 3) and train_a.shape == (7200, 1)
assert set(episode_seeds[:60]).isdisjoint(set(episode_seeds[75:]))
assert torch.allclose(train_s[:, :2].square().sum(1).mean(), torch.tensor(1.0), atol=1e-5)
print({"episode split": (60, 15, 15), "transition split": (len(train_s), len(val_s), len(test_s)), "reward range": (round(float(train_r.min()), 3), round(float(train_r.max()), 3))})
```

</details>

划分发生在任何可学习预处理或模型拟合之前。同一 episode 中相邻 transition 高度相关；如果随机拆分单行记录，几乎相同的状态就可能同时进入训练集和测试集，造成泄漏。


### **序列决策与马尔可夫决策过程** {#sequential-decisions-mdps}

马尔可夫决策过程（MDP）是一个元组：

$$
\mathcal{M}=(\mathcal{S},\mathcal{A},P,R,\gamma,\rho_0).
$$

$\mathcal{S}$ 与 $\mathcal{A}$ 分别是状态空间和动作空间；$P(s'|s,a)$ 是转移分布；$R(s,a,s')$ 指定奖励；$\gamma\in[0,1)$ 对延迟奖励进行折扣；$\rho_0$ 是初始状态分布。马尔可夫假设认为，当前状态包含预测下一状态和奖励所需的全部信息：

$$
p(s_{t+1},r_t\mid s_0,a_0,\ldots,s_t,a_t)=p(s_{t+1},r_t\mid s_t,a_t).
$$

这是对**状态表示**的假设，不一定适用于原始观测。单张相机画面可能隐藏速度或视野之外的物体，从而形成部分可观测 MDP（POMDP）。帧堆叠、循环状态、信念状态推断或世界模型都可以补充有用记忆。

每一步中，策略 $\pi_\theta(a|s)$ 产生一个动作分布，环境随后生成 $r_t,s_{t+1}$。由于 $\pi_\theta$ 会影响智能体访问哪些状态，RL 训练集是内生的：策略改善或失稳都会改变数据分布。

<details>
<summary><strong>Python：追踪状态、动作、奖励、任务终止与时间截断</strong></summary>

```python
trace_env = LocalPendulum(horizon=12)
observation = trace_env.reset(seed=1720)
trajectory = []
for step in range(12):
    action = scripted_action(observation, np.random.default_rng(1720 + step))
    next_observation, reward, terminated, truncated = trace_env.step(action)
    trajectory.append({"t": step, "state": observation.copy(), "action": action, "reward": reward, "terminated": terminated, "truncated": truncated})
    observation = next_observation

assert not any(item["terminated"] for item in trajectory)
assert trajectory[-1]["truncated"] and not trajectory[-2]["truncated"]
assert all(item["state"].shape == (3,) for item in trajectory)
print({"first transition": {"state": np.round(trajectory[0]["state"], 3).tolist(), "action": round(trajectory[0]["action"], 3), "reward": round(trajectory[0]["reward"], 3)}, "last step truncated": trajectory[-1]["truncated"]})
```

</details>

`terminated` 表示底层任务到达终止状态；`truncated` 表示外部限制停止了数据收集。把每个时间上限都当成真正终止会强制价值目标变为零，从而使持续任务的估计产生偏差。有限时域建模可以把剩余时间纳入状态来消除歧义；否则必须明确说明是否在截断位置继续自举。


### **回报、价值函数与 Bellman 方程** {#returns-values-bellman}

从时刻 $t$ 开始的折扣回报为

$$
G_t=\sum_{k=0}^{T-t-1}\gamma^k r_{t+k}.
$$

状态价值 $V^\pi(s)=\mathbb{E}_\pi[G_t|s_t=s]$ 衡量随后遵循 $\pi$ 时的期望回报。动作价值 $Q^\pi(s,a)$ 先以某个动作作为条件，再遵循该策略。二者之差 $A^\pi(s,a)=Q^\pi(s,a)-V^\pi(s)$ 称为优势（advantage）：它表示当前动作相对该策略在此状态下通常行为的好坏。

![一步 Bellman 目标把即时奖励与自举得到的未来价值结合起来。](assets/dl17-bellman-backup.svg){fig-align="center" width="76%" fig-alt="当前估计与由奖励加下一状态折扣价值构成的目标进行比较。"}

Bellman 期望方程是递归一致性条件：

$$
V^\pi(s)=\mathbb{E}_{a\sim\pi,s'\sim P}[r+\gamma V^\pi(s')],\qquad Q^\pi(s,a)=\mathbb{E}_{s'\sim P}[r+\gamma\mathbb{E}_{a'\sim\pi}Q^\pi(s',a')].
$$

Monte Carlo 学习等待完整 $G_t$：自举偏差较低，但方差较高且更新延迟。时序差分（TD）学习使用 $r_t+\gamma V(s_{t+1})$：能够更早更新且方差较低，但会继承当前 critic 的偏差。多步回报与广义优势估计（GAE）在二者之间进行插值。

<details>
<summary><strong>Python：比较 reward-to-go、一步 TD 与 GAE</strong></summary>

```python
def discounted_returns(rewards, gamma=0.98):
    output = torch.zeros_like(rewards)
    running = torch.tensor(0.0)
    for index in range(len(rewards) - 1, -1, -1):
        running = rewards[index] + gamma * running
        output[index] = running
    return output


def generalized_advantages(rewards, values, next_values, gamma=0.98, lam=0.95):
    deltas = rewards + gamma * next_values - values
    advantages = torch.zeros_like(deltas)
    running = torch.tensor(0.0)
    for index in range(len(deltas) - 1, -1, -1):
        running = deltas[index] + gamma * lam * running
        advantages[index] = running
    return advantages


example_rewards = torch.tensor([item["reward"] for item in trajectory], dtype=torch.float32)
proxy_values = -2.0 * torch.arange(len(example_rewards), 0, -1, dtype=torch.float32)
proxy_next = torch.cat([proxy_values[1:], torch.zeros(1)])
returns = discounted_returns(example_rewards)
td_residuals = example_rewards + 0.98 * proxy_next - proxy_values
gae = generalized_advantages(example_rewards, proxy_values, proxy_next)
assert returns.shape == td_residuals.shape == gae.shape
assert torch.allclose(returns[-1], example_rewards[-1])
print({"G_0": round(float(returns[0]), 3), "mean absolute TD residual": round(float(td_residuals.abs().mean()), 3), "mean absolute GAE": round(float(gae.abs().mean()), 3)})
```

</details>

上面的代理价值有意设置得不完美，因此 TD residual 不为零。在真实智能体中 critic 也是学习得到的，所以它的误差会进入 actor 的目标。这种耦合解释了为什么 RL 诊断必须同时查看回报、价值损失、解释方差、熵和动作分布。


### **深度 Q 网络** {#deep-q-networks}

Q-learning 寻找 $Q^*(s,a)=\mathbb{E}[r+\gamma\max_{a'}Q^*(s',a')]$。深度 Q 网络（DQN）使用神经网络表示 $Q_\theta(s,a)$，并针对以下目标最小化 Huber 或平方误差：

$$
y=r+\gamma(1-d)\max_{a'}Q_{\bar\theta}(s',a').
$$

$d$ 标记终止 transition，$\bar\theta$ 表示滞后的 target network。这里同时出现三种风险：函数逼近会把更新推广到其他状态；自举会使用另一个估计作为训练目标；off-policy replay 包含旧策略产生的动作。这一组合可能发散。

![Replay 与滞后的 target network 缓解了 DQN 中两种移动目标效应。](assets/dl17-dqn-stabilizers.svg){fig-align="center" width="78%" fig-alt="DQN 训练管线连接在线 Q 网络、replay buffer、目标 Q 网络与 TD 损失。"}

Experience replay 降低时间相关性并复用样本。Target network 的变化速度慢于在线网络。奖励缩放、梯度裁剪、Huber loss、Double DQN target、dueling head 和 prioritized replay 分别处理其他失败模式。DQN 天然假设动作集合有限，因此本例把扭矩离散为五个值。

<details>
<summary><strong>PyTorch：在离散化 Pendulum 扭矩上训练紧凑 DQN</strong></summary>

```python
class QNetwork(nn.Module):
    def __init__(self, action_count=5):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, action_count))

    def forward(self, state):
        return self.net(state)


seed_everything(1730)
discrete_torques = torch.linspace(-2, 2, 5)
dqn = QNetwork()
dqn_target = copy.deepcopy(dqn).eval()
dqn_optimizer = torch.optim.AdamW(dqn.parameters(), lr=1e-3)
replay = deque(maxlen=6000)
dqn_env = LocalPendulum(horizon=120)
state = dqn_env.reset(seed=1730)
rng = np.random.default_rng(1730)
dqn_losses = []

for global_step in range(5200):
    epsilon = max(0.08, 1.0 - global_step / 4200)
    if rng.random() < epsilon:
        action_index = int(rng.integers(5))
    else:
        with torch.no_grad():
            action_index = int(dqn(torch.tensor(state).unsqueeze(0)).argmax(1))
    next_state, reward, _, truncated = dqn_env.step(float(discrete_torques[action_index]))
    replay.append((state, action_index, reward / 10.0, next_state, truncated))
    state = next_state
    if truncated:
        state = dqn_env.reset(seed=1730 + global_step)

    if len(replay) >= 256:
        batch_ids = rng.integers(0, len(replay), size=64)
        batch = [replay[int(i)] for i in batch_ids]
        bs = torch.tensor(np.stack([x[0] for x in batch]), dtype=torch.float32)
        ba = torch.tensor([x[1] for x in batch], dtype=torch.long)
        br = torch.tensor([x[2] for x in batch], dtype=torch.float32)
        bns = torch.tensor(np.stack([x[3] for x in batch]), dtype=torch.float32)
        bd = torch.tensor([x[4] for x in batch], dtype=torch.float32)
        prediction = dqn(bs).gather(1, ba[:, None]).squeeze(1)
        with torch.no_grad():
            target = br + 0.98 * (1 - bd) * dqn_target(bns).max(1).values
        loss = F.smooth_l1_loss(prediction, target)
        dqn_optimizer.zero_grad(); loss.backward()
        nn.utils.clip_grad_norm_(dqn.parameters(), 10.0)
        dqn_optimizer.step()
        dqn_losses.append(float(loss.detach()))
    if global_step % 160 == 0:
        dqn_target.load_state_dict(dqn.state_dict())


def dqn_policy(observation):
    with torch.no_grad():
        index = int(dqn(torch.tensor(observation, dtype=torch.float32).unsqueeze(0)).argmax(1))
    return float(discrete_torques[index])


assert len(replay) == 5200 and np.isfinite(dqn_losses[-1])
print({"final mean TD loss": round(float(np.mean(dqn_losses[-100:])), 4), "replay transitions": len(replay), "learned start-state torque": round(dqn_policy(np.array([1.0, 0.0, 0.0], dtype=np.float32)), 2)})
```

</details>

当前实现把教学实验的时域上限视为终止。对于因外部时间限制而截断的持续任务，目标通常应当继续自举。由于同一组带噪估计同时选择和评估 $\max_{a'}$，Q 值还可能系统性高估；Double DQN 会拆分这两个角色。


### **策略梯度方法** {#policy-gradient-methods}

基于价值的控制选择估计价值最高的动作。策略梯度方法则直接优化可微策略，因此能够处理随机行为和连续动作。对于 $J(\theta)=\mathbb{E}_{\tau\sim\pi_\theta}[G(\tau)]$，策略梯度定理给出

$$
\nabla_\theta J(\theta)=\mathbb{E}_{s,a\sim\pi_\theta}[\nabla_\theta\log\pi_\theta(a|s)Q^{\pi_\theta}(s,a)].
$$

对数导数把依赖环境的轨迹概率转化为策略概率的梯度，环境动力学本身不必可微。当 baseline 与动作无关时，用 $Q-b(s)$ 替换 $Q$ 不会改变期望梯度，但良好的 baseline 可以显著降低方差。

![每个采样动作都会收到按估计优势加权的概率更新。](assets/dl17-policy-gradient.svg){fig-align="center" width="78%" fig-alt="状态与动作构成的轨迹最终得到回报，对数策略梯度由优势加权。"}

REINFORCE 使用采样得到的 reward-to-go。在采样假设成立时，它简单且无偏，但一条很差的轨迹就可能主导一次更新。优势归一化会改变有限 batch 的尺度；熵奖励延缓策略过早坍缩；梯度裁剪限制极端步长。这些方法都不能替代多个随机种子。

<details>
<summary><strong>PyTorch：训练 REINFORCE 并检查 baseline 中心化</strong></summary>

```python
class CategoricalActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 48), nn.Tanh(), nn.Linear(48, 5))

    def forward(self, states):
        return torch.distributions.Categorical(logits=self.net(states))


def categorical_episode(actor, seed, horizon=120):
    env = LocalPendulum(horizon=horizon)
    state = env.reset(seed)
    states, actions, rewards = [], [], []
    previous_rng = torch.random.get_rng_state()
    torch.manual_seed(seed)
    for _ in range(horizon):
        state_tensor = torch.tensor(state, dtype=torch.float32)
        action_index = int(actor(state_tensor).sample())
        next_state, reward, _, truncated = env.step(float(discrete_torques[action_index]))
        states.append(state_tensor); actions.append(action_index); rewards.append(reward / 10.0)
        state = next_state
        if truncated:
            break
    torch.random.set_rng_state(previous_rng)
    return torch.stack(states), torch.tensor(actions), torch.tensor(rewards, dtype=torch.float32)


seed_everything(1740)
reinforce_actor = CategoricalActor()
reinforce_optimizer = torch.optim.Adam(reinforce_actor.parameters(), lr=2e-3)
reinforce_trace = []
for update in range(34):
    collected = [categorical_episode(reinforce_actor, 1740 + update * 7 + i) for i in range(4)]
    states = torch.cat([x[0] for x in collected])
    actions = torch.cat([x[1] for x in collected])
    returns_batch = torch.cat([discounted_returns(x[2]) for x in collected])
    advantages = (returns_batch - returns_batch.mean()) / (returns_batch.std() + 1e-6)
    distribution = reinforce_actor(states)
    loss = -(distribution.log_prob(actions) * advantages).mean() - 0.002 * distribution.entropy().mean()
    reinforce_optimizer.zero_grad(); loss.backward(); reinforce_optimizer.step()
    reinforce_trace.append(float(returns_batch[:120].mean()))

probe_states, probe_actions, probe_rewards = categorical_episode(reinforce_actor, 1799)
probe_returns = discounted_returns(probe_rewards)
centered_weights = probe_returns - probe_returns.mean()
assert abs(float(centered_weights.mean())) < 1e-5
print({"updates": len(reinforce_trace), "last return-to-go mean": round(reinforce_trace[-1], 3), "raw weight mean": round(float(probe_returns.mean()), 3), "centered weight mean": round(float(centered_weights.mean()), 6)})
```

</details>

常数 baseline 可以使 batch 权重中心化，却无法解释不同状态之间的可预测变化。学习得到的价值 baseline 会预测哪些状态天然具有更大的未来回报，由此引出 actor-critic 方法。


### **Actor-Critic 学习** {#actor-critic-learning}

Actor-critic 智能体同时维护策略（actor）和价值估计器（critic）。一步 TD residual

$$
\delta_t=r_t+\gamma V_\phi(s_{t+1})-V_\phi(s_t)
$$

既是 critic 的误差，也是低成本的优势估计。当 $\delta_t>0$ 时，actor 提高 $\log\pi_\theta(a_t|s_t)$；当 $\delta_t<0$ 时则降低它，而 critic 最小化回归损失。

![Actor 改变行为，critic 则把奖励与未来价值转化为优势信号。](assets/dl17-actor-critic.svg){fig-align="center" width="76%" fig-alt="状态特征分支进入 actor 与 critic，二者输出在 TD advantage 中汇合。"}

共享 encoder 可以减少计算，却会把策略损失与价值损失的梯度耦合起来。学习过慢的 critic 会产生带噪优势；过拟合的 critic 会产生带有置信度的偏差。常见诊断包括价值损失、预测价值尺度、解释方差、策略熵、梯度范数以及 actor/critic 学习率敏感性。

<details>
<summary><strong>PyTorch：联合更新分类 actor 与状态价值 critic</strong></summary>

```python
class ActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(3, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.actor = nn.Linear(64, 5)
        self.critic = nn.Linear(64, 1)

    def forward(self, states):
        features = self.trunk(states)
        return torch.distributions.Categorical(logits=self.actor(features)), self.critic(features).squeeze(-1)


class ActorView:
    def __call__(self, states):
        return a2c(states)[0]


seed_everything(1750)
a2c = ActorCritic()
a2c_optimizer = torch.optim.Adam(a2c.parameters(), lr=1.5e-3)
a2c_value_losses = []
for episode_id in range(90):
    states, actions, rewards = categorical_episode(ActorView(), 1750 + episode_id)
    returns_batch = discounted_returns(rewards)
    distribution, values = a2c(states)
    advantages = returns_batch - values.detach()
    actor_loss = -(distribution.log_prob(actions) * advantages).mean()
    critic_loss = F.smooth_l1_loss(values, returns_batch)
    loss = actor_loss + 0.5 * critic_loss - 0.003 * distribution.entropy().mean()
    a2c_optimizer.zero_grad(); loss.backward()
    nn.utils.clip_grad_norm_(a2c.parameters(), 5.0)
    a2c_optimizer.step()
    a2c_value_losses.append(float(critic_loss.detach()))


def a2c_policy(observation):
    with torch.no_grad():
        distribution, _ = a2c(torch.tensor(observation, dtype=torch.float32))
    return float(discrete_torques[int(distribution.probs.argmax())])


assert np.isfinite(a2c_value_losses).all()
print({"last critic loss": round(a2c_value_losses[-1], 3), "median last-20 critic loss": round(float(np.median(a2c_value_losses[-20:])), 3), "upright torque": round(a2c_policy(np.array([1.0, 0.0, 0.0], dtype=np.float32)), 2)})
```

</details>

尽管 TD residual 提供了 actor-critic 的概念动机，这个紧凑实现仍使用 Monte Carlo return 作为较稳定的教学目标。生产级 A2C/A3C 通常收集固定长度 rollout 片段，并在片段边界使用 critic 继续自举。


### **近端策略优化** {#proximal-policy-optimization}

在同一个 on-policy batch 上执行多个 epoch 可以提高样本利用率，但第一次更新以后，数据已经来自旧策略。PPO 测量 $r_t(\theta)=\pi_\theta(a_t|s_t)/\pi_{\theta_{\mathrm{old}}}(a_t|s_t)$，并优化

$$
L^{\mathrm{clip}}(\theta)=\mathbb{E}_t[\min(r_t(\theta)\hat A_t,\operatorname{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\hat A_t)].
$$

![PPO 会消除概率比率越过裁剪区间后继续移动的激励。](assets/dl17-ppo-clipping.svg){fig-align="center" width="76%" fig-alt="分段 surrogate 曲线展示概率比率在一减 epsilon 与一加 epsilon 附近的裁剪。"}

当优势为正时，把所选动作概率提高到 $1+\epsilon$ 以上不会带来额外 surrogate 收益；当优势为负时，把它降低到 $1-\epsilon$ 以下同样会被裁剪。PPO **并不**保证严格的 trust region：裁剪只影响采样目标项，因此仍要监控近似 KL divergence、clip fraction、熵与价值误差。

<details>
<summary><strong>PyTorch：收集 rollout、计算 GAE 并应用 PPO clipping</strong></summary>

```python
class GaussianActorCritic(nn.Module):
    def __init__(self):
        super().__init__()
        self.trunk = nn.Sequential(nn.Linear(3, 64), nn.Tanh(), nn.Linear(64, 64), nn.Tanh())
        self.mean = nn.Linear(64, 1)
        self.value = nn.Linear(64, 1)
        self.log_std = nn.Parameter(torch.tensor([-0.35]))

    def distribution_value(self, states):
        features = self.trunk(states)
        return torch.distributions.Normal(self.mean(features), self.log_std.exp()), self.value(features).squeeze(-1)

    @staticmethod
    def log_prob(distribution, raw_action):
        squashed = torch.tanh(raw_action)
        return (distribution.log_prob(raw_action) - torch.log(1 - squashed.square() + 1e-6)).sum(-1)


def collect_ppo_episode(model, seed):
    env, state = LocalPendulum(horizon=120), None
    state = env.reset(seed)
    states, raw_actions, log_probs, rewards, values = [], [], [], [], []
    torch.manual_seed(seed)
    for _ in range(120):
        state_tensor = torch.tensor(state, dtype=torch.float32)
        with torch.no_grad():
            distribution, value = model.distribution_value(state_tensor)
            raw_action = distribution.sample()
            log_prob = model.log_prob(distribution, raw_action)
        next_state, reward, _, truncated = env.step(float(2.0 * torch.tanh(raw_action)))
        states.append(state_tensor); raw_actions.append(raw_action); log_probs.append(log_prob)
        rewards.append(reward / 10.0); values.append(value)
        state = next_state
        if truncated:
            break
    rewards = torch.tensor(rewards, dtype=torch.float32)
    values = torch.stack(values)
    next_values = torch.cat([values[1:], torch.zeros(1)])
    advantages = generalized_advantages(rewards, values, next_values, gamma=0.98, lam=0.95)
    return torch.stack(states), torch.stack(raw_actions), torch.stack(log_probs), advantages, advantages + values


seed_everything(1760)
ppo = GaussianActorCritic()
ppo_optimizer = torch.optim.Adam(ppo.parameters(), lr=1.2e-3)
ppo_clip_fractions = []
for update in range(16):
    batches = [collect_ppo_episode(ppo, 1760 + update * 5 + offset) for offset in range(4)]
    states = torch.cat([x[0] for x in batches])
    raw_actions = torch.cat([x[1] for x in batches])
    old_log_probs = torch.cat([x[2] for x in batches])
    advantages = torch.cat([x[3] for x in batches])
    value_targets = torch.cat([x[4] for x in batches])
    advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-6)
    for _ in range(4):
        distribution, values = ppo.distribution_value(states)
        new_log_probs = ppo.log_prob(distribution, raw_actions)
        ratio = (new_log_probs - old_log_probs).exp()
        policy_loss = -torch.minimum(ratio * advantages, ratio.clamp(0.8, 1.2) * advantages).mean()
        value_loss = F.smooth_l1_loss(values, value_targets)
        total_loss = policy_loss + 0.5 * value_loss - 0.002 * distribution.entropy().sum(-1).mean()
        ppo_optimizer.zero_grad(); total_loss.backward()
        nn.utils.clip_grad_norm_(ppo.parameters(), 1.0)
        ppo_optimizer.step()
    ppo_clip_fractions.append(float(((ratio - 1).abs() > 0.2).float().mean()))


def ppo_policy(observation):
    with torch.no_grad():
        distribution, _ = ppo.distribution_value(torch.tensor(observation, dtype=torch.float32))
    return float(2.0 * torch.tanh(distribution.mean))


assert 0 <= ppo_clip_fractions[-1] <= 1
print({"updates": 16, "last clip fraction": round(ppo_clip_fractions[-1], 3), "learned log standard deviation": round(float(ppo.log_std.detach()), 3)})
```

</details>

代码保存 `tanh` 之前的动作，使新旧 log probability 使用相同的变量变换校正。漏掉该 Jacobian 会在无提示的情况下优化错误的有界动作密度。其他常见错误包括让梯度流入旧 log probability、没有归一化观测，以及在时间上限处错误自举。


### **Soft Actor-Critic** {#soft-actor-critic}

Soft Actor-Critic（SAC）是一种面向连续动作的 off-policy actor-critic 方法。它同时最大化奖励和策略熵：

$$
J(\pi)=\mathbb{E}_{\tau\sim\pi}\left[\sum_t\gamma^t(r_t+\alpha\mathcal{H}(\pi(\cdot|s_t)))\right].
$$

$\alpha$ 是 temperature。较大的值偏向多样动作，较小的值偏向利用。Soft target 为

$$
y=r+\gamma(1-d)\left[\min_iQ_{\bar\phi_i}(s',a')-\alpha\log\pi_\theta(a'|s')\right],\quad a'\sim\pi_\theta(\cdot|s').
$$

![SAC 使用 replay、双 critic、压缩随机 actor 与熵正则化。](assets/dl17-sac-objective.svg){fig-align="center" width="76%" fig-alt="Replay 数据进入双 critic 与 squashed Gaussian actor，并在最大熵目标中结合。"}

双 critic 使用较小估计来降低乐观误差。Actor 使用重参数化 $u=\mu_\theta(s)+\sigma_\theta(s)\epsilon$、$a=2\tanh u$，从而让梯度穿过采样动作。Target critic 缓慢更新。自动 temperature 调节可以匹配目标熵，而不是固定 $\alpha$。

<details>
<summary><strong>PyTorch：在共享 replay 上审查 SAC critic 与 actor 目标</strong></summary>

```python
class SquashedGaussianActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.body = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU())
        self.mean, self.log_std = nn.Linear(64, 1), nn.Linear(64, 1)

    def sample(self, states):
        features = self.body(states)
        mean = self.mean(features)
        log_std = self.log_std(features).clamp(-5, 1)
        distribution = torch.distributions.Normal(mean, log_std.exp())
        raw = distribution.rsample()
        squashed = torch.tanh(raw)
        action = 2.0 * squashed
        log_prob = distribution.log_prob(raw) - torch.log(2.0 * (1 - squashed.square()) + 1e-6)
        return action, log_prob.sum(-1, keepdim=True)


class ContinuousQ(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, states, actions):
        return self.net(torch.cat([states, actions], dim=-1))


seed_everything(1770)
sac_actor, sac_q1, sac_q2 = SquashedGaussianActor(), ContinuousQ(), ContinuousQ()
sac_t1, sac_t2 = copy.deepcopy(sac_q1), copy.deepcopy(sac_q2)
sac_actor_opt = torch.optim.Adam(sac_actor.parameters(), lr=8e-4)
sac_q_opt = torch.optim.Adam(list(sac_q1.parameters()) + list(sac_q2.parameters()), lr=8e-4)
sac_rng, alpha = np.random.default_rng(1770), 0.15

for update in range(650):
    ids = torch.tensor(sac_rng.integers(0, len(train_s), size=128))
    states, actions = train_s[ids], train_a[ids]
    rewards, next_states, dones = train_r[ids] / 10.0, train_ns[ids], train_done[ids]
    with torch.no_grad():
        next_actions, next_log_prob = sac_actor.sample(next_states)
        soft_next_q = torch.minimum(sac_t1(next_states, next_actions), sac_t2(next_states, next_actions))
        q_target = rewards + 0.98 * (1 - dones) * (soft_next_q - alpha * next_log_prob)
    q_loss = F.mse_loss(sac_q1(states, actions), q_target) + F.mse_loss(sac_q2(states, actions), q_target)
    sac_q_opt.zero_grad(); q_loss.backward(); sac_q_opt.step()

    sampled_actions, log_prob = sac_actor.sample(states)
    actor_loss = (alpha * log_prob - torch.minimum(sac_q1(states, sampled_actions), sac_q2(states, sampled_actions))).mean()
    sac_actor_opt.zero_grad(); actor_loss.backward(); sac_actor_opt.step()
    with torch.no_grad():
        for target_parameter, parameter in zip(sac_t1.parameters(), sac_q1.parameters()):
            target_parameter.mul_(0.995).add_(parameter, alpha=0.005)
        for target_parameter, parameter in zip(sac_t2.parameters(), sac_q2.parameters()):
            target_parameter.mul_(0.995).add_(parameter, alpha=0.005)


def sac_policy(observation):
    with torch.no_grad():
        features = sac_actor.body(torch.tensor(observation, dtype=torch.float32))
        return float(2.0 * torch.tanh(sac_actor.mean(features)))


assert torch.isfinite(q_loss) and torch.isfinite(actor_loss)
print({"critic loss": round(float(q_loss.detach()), 4), "actor loss": round(float(actor_loss.detach()), 4), "sample action range": (round(float(sampled_actions.min().detach()), 3), round(float(sampled_actions.max().detach()), 3))})
```

</details>

本例复用本章的**固定日志数据集**，因此不是标准在线 SAC benchmark。在线 SAC 会交替执行环境交互和 replay 更新，使 buffer 跟随不断改善的策略。在固定数据集上，actor 动作可能离开数据支持；离线 RL 小节解释了为什么不受约束的 SAC 目标可能利用 critic 误差。


### **探索与信用分配** {#exploration-credit-assignment}

探索要回答“收集哪些经验”，信用分配要回答“哪些过去决策导致了后续结果”。二者相互影响：智能体无法为从未尝试的行为分配信用，而稀疏的延迟奖励也很难指导下一步应该尝试什么。

动作空间策略包括 $\epsilon$-greedy 探索、策略熵、参数噪声、乐观不确定性和具有时间相关性的控制噪声。状态空间策略包括计数、预测误差、信息增益和课程学习。当随机观测始终不可预测时，新颖度信号可能失败，这就是“noisy TV”问题。

$n$ 步目标为 $G_t^{(n)}=\sum_{k=0}^{n-1}\gamma^k r_{t+k}+\gamma^nV(s_{t+n})$。GAE 对 TD residual 进行指数加权求和：

$$
\hat A_t^{\mathrm{GAE}(\gamma,\lambda)}=\sum_{l=0}^{\infty}(\gamma\lambda)^l\delta_{t+l}.
$$

<details>
<summary><strong>Python：检查 GAE lambda 如何改变时间信用</strong></summary>

```python
credit_rewards = torch.tensor([0.0, 0.0, 0.0, 0.0, 1.0])
credit_values = torch.tensor([0.15, 0.18, 0.22, 0.30, 0.40])
credit_next = torch.cat([credit_values[1:], torch.zeros(1)])
gae_by_lambda = {lam: generalized_advantages(credit_rewards, credit_values, credit_next, gamma=0.99, lam=lam) for lam in (0.0, 0.5, 0.95, 1.0)}
assert torch.allclose(gae_by_lambda[0.0], credit_rewards + 0.99 * credit_next - credit_values)
print({f"lambda={lam}": [round(float(x), 3) for x in advantages] for lam, advantages in gae_by_lambda.items()})
```

</details>

当 $\lambda=0$ 时，只有即时 TD residual 获得信用；当 $\lambda=1$ 时，后续 residual 会沿整个后缀传播。最佳取值取决于 critic 精度、时域、奖励延迟与 batch size。Reward shaping 必须保持预期目标；容易优化的稠密代理可能产生 reward hacking，而不是有效探索。


### **模仿学习与离线强化学习** {#imitation-offline-reinforcement-learning}

Behavioral cloning（BC）最小化 $\mathcal{L}_{\mathrm{BC}}=-\mathbb{E}_{(s,a)\sim\mathcal D}\log\pi_\theta(a|s)$。部署时的错误可能把策略带到 demonstration 中从未出现的状态，之后错误会继续累积。DAgger 可以向 expert 查询 learner 访问过的状态，但数据收集固定时无法使用这种选择。

离线 RL 尝试只用静态数据改进策略。标准 Bellman backup 会评估学习策略提出的动作，其中包括 $\mathcal D$ 中表示不足的动作。函数逼近可能为这些动作赋予虚假的大价值。

![离线 critic 可能把日志支持范围之外的外推误差误认为高价值动作。](assets/dl17-offline-shift.svg){fig-align="center" width="76%" fig-alt="日志中的状态动作点只占据一个区域，而学习策略移动到该区域之外缺乏支持的动作。"}

Conservative Q-learning 添加惩罚项。离散形式为

$$
\mathcal R_{\mathrm{CQL}}=\mathbb E_{s\sim\mathcal D}\left[\log\sum_a\exp Q(s,a)-Q(s,a_{\mathcal D})\right].
$$

[CQL 论文](https://proceedings.neurips.cc/paper/2020/hash/0d2b2061826a5df3221116a5085a6052-Abstract.html)研究了分布偏移下的保守价值估计。过强的保守性也可能阻止策略超越 behavior policy。

<details>
<summary><strong>PyTorch：behavioral cloning 与连续动作 CQL penalty</strong></summary>

```python
class DeterministicActor(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(3, 64), nn.ReLU(), nn.Linear(64, 64), nn.ReLU(), nn.Linear(64, 1))

    def forward(self, states):
        return 2.0 * torch.tanh(self.net(states))


seed_everything(1780)
bc_actor = DeterministicActor()
bc_optimizer = torch.optim.AdamW(bc_actor.parameters(), lr=1e-3, weight_decay=1e-4)
for _ in range(500):
    ids = torch.randint(len(train_s), (128,))
    bc_loss = F.mse_loss(bc_actor(train_s[ids]), train_a[ids])
    bc_optimizer.zero_grad(); bc_loss.backward(); bc_optimizer.step()

with torch.no_grad():
    validation_bc_mse = F.mse_loss(bc_actor(val_s), val_a)
    sampled_random_actions = torch.empty(len(val_s), 12, 1).uniform_(-2, 2)
    repeated_states = val_s[:, None, :].expand(-1, 12, -1)
    random_q = sac_q1(repeated_states.reshape(-1, 3), sampled_random_actions.reshape(-1, 1)).reshape(len(val_s), 12)
    data_q = sac_q1(val_s, val_a).squeeze(1)
    cql_penalty = torch.logsumexp(random_q, dim=1).mean() - data_q.mean()


def bc_policy(observation):
    with torch.no_grad():
        return float(bc_actor(torch.tensor(observation, dtype=torch.float32)))


assert validation_bc_mse >= 0 and torch.isfinite(cql_penalty)
print({"BC validation action MSE": round(float(validation_bc_mse), 4), "CQL support penalty": round(float(cql_penalty), 4), "logged action std": round(float(train_a.std()), 3)})
```

</details>

随机动作的 log-sum-exp 是连续动作下的 Monte Carlo 近似，并不是完整 CQL 实现。可靠的离线研究应描述覆盖度、behavior 质量、episode 边界、奖励重标记，以及模型选择是否使用了在线交互。调参期间访问测试环境可能悄悄破坏“离线”这一前提。


### **基于模型的强化学习** {#model-based-reinforcement-learning}

基于模型的 RL 学习或使用 $\hat p_\psi(s_{t+1},r_t|s_t,a_t)$，随后进行规划、生成合成经验或通过模型求导。模型可以重复使用每条真实 transition。它的核心代价是**模型偏差**：策略可能访问学习动力学不准确的状态，并利用这些误差。

![较小的一步预测误差可能累积成较大的轨迹误差。](assets/dl17-model-rollout-bias.svg){fig-align="center" width="76%" fig-alt="真实轨迹与学习轨迹从同一点出发，但随着 rollout horizon 增大而分离。"}

Ensemble 通过不同初始化或重采样来近似 epistemic uncertainty，规划时可以惩罚模型分歧。[MBPO](https://proceedings.neurips.cc/paper/2019/hash/5faf461eff3099671ad63c6f3f094f7f-Abstract.html) 研究的短模型 rollout 从 replay 中的真实状态分支出去，可以在增加合成 transition 的同时限制误差累积。

<details>
<summary><strong>PyTorch：拟合动力学 ensemble，并按时域测量 rollout error</strong></summary>

```python
class DynamicsModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 96), nn.SiLU(), nn.Linear(96, 96), nn.SiLU(), nn.Linear(96, 4))


input_mean = torch.cat([train_s, train_a], dim=1).mean(0)
input_std = torch.cat([train_s, train_a], dim=1).std(0).clamp_min(1e-4)
target_train = torch.cat([train_ns - train_s, train_r / 10.0], dim=1)
target_mean, target_std = target_train.mean(0), target_train.std(0).clamp_min(1e-4)


def normalized_dynamics(model, states, actions):
    normalized_input = (torch.cat([states, actions], dim=1) - input_mean) / input_std
    prediction = model.net(normalized_input) * target_std + target_mean
    return states + prediction[:, :3], prediction[:, 3:4]


ensemble = []
for model_id in range(3):
    seed_everything(1790 + model_id)
    model = DynamicsModel()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
    generator = torch.Generator().manual_seed(1790 + model_id)
    for _ in range(550):
        ids = torch.randint(len(train_s), (160,), generator=generator)
        normalized_input = (torch.cat([train_s[ids], train_a[ids]], 1) - input_mean) / input_std
        normalized_target = (target_train[ids] - target_mean) / target_std
        loss = F.mse_loss(model.net(normalized_input), normalized_target)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
    ensemble.append(model.eval())

with torch.no_grad():
    next_predictions = torch.stack([normalized_dynamics(model, val_s, val_a)[0] for model in ensemble])
    one_step_mse = F.mse_loss(next_predictions.mean(0), val_ns)
    disagreement = next_predictions.var(0).mean()

heldout = test_episodes[0]
predicted = torch.tensor(heldout[0][0], dtype=torch.float32).unsqueeze(0)
rollout_errors = []
with torch.no_grad():
    for transition in heldout[:40]:
        action = torch.tensor([[transition[1]]], dtype=torch.float32)
        candidates = torch.stack([normalized_dynamics(model, predicted, action)[0] for model in ensemble])
        predicted = candidates.mean(0)
        true_next = torch.tensor(transition[3], dtype=torch.float32).unsqueeze(0)
        rollout_errors.append(float(F.mse_loss(predicted, true_next)))

assert rollout_errors[-1] >= 0 and torch.isfinite(one_step_mse)
print({"validation one-step MSE": round(float(one_step_mse), 5), "ensemble disagreement": round(float(disagreement), 6), "open-loop MSE": {"h=1": round(rollout_errors[0], 5), "h=10": round(rollout_errors[9], 5), "h=40": round(rollout_errors[39], 5)}})
```

</details>

一步验证误差是必要但不充分的：规划会改变状态—动作分布，而重复预测会把模型输出重新作为输入。有效诊断包括按时域分解的误差、ensemble disagreement、校准、约束违例，以及候选策略在真实环境和模型中的回报差距。


### **世界模型** {#world-models}

世界模型学习用于规划或策略学习的紧凑预测状态。最初的 [World Models](https://worldmodels.github.io/) 系统把视觉 $V$、循环记忆 $M$ 与控制器 $C$ 分开。现代 latent-imagination 智能体（如 [DreamerV3](https://arxiv.org/abs/2301.04104)）会联合学习表示、潜在动力学、奖励和 continuation predictor，然后在想象轨迹上训练 actor-critic 组件。

![原始 World Models 架构把视觉、记忆与控制器组件分开。](assets/dl17-world-model-overview.svg){fig-align="center" width="74%" fig-alt="World Models 总览图包含视觉 encoder、循环记忆模型、控制器与环境。"}

*来源：Ha 与 Schmidhuber，[World Models](https://worldmodels.github.io/)，图由作者按照 [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) 许可发布。*

循环状态空间模型（RSSM）通常结合确定性记忆 $h_t$ 与随机状态 $z_t$：

$$
h_t=f_\psi(h_{t-1},z_{t-1},a_{t-1}),\qquad p_\psi(z_t|h_t),\qquad q_\psi(z_t|h_t,o_t).
$$

训练期间 posterior $q$ 会纳入真实观测；想象期间 prior $p$ 则在没有真实观测时预测。Decoder 预测观测或任务相关特征、奖励和 continuation，KL 项使 posterior 与 prior 对齐。通过想象潜在轨迹得到 actor gradient 可以提高效率，但 actor 也可能利用模型缺陷。

<details>
<summary><strong>PyTorch：训练紧凑循环世界模型并测试开环想象</strong></summary>

```python
def episode_windows(selected, length=20):
    states, actions, next_states, rewards = [], [], [], []
    for episode in selected:
        for start in range(0, len(episode) - length + 1, length):
            window = episode[start:start + length]
            states.append(np.stack([x[0] for x in window]))
            actions.append(np.array([[x[1]] for x in window], dtype=np.float32))
            next_states.append(np.stack([x[3] for x in window]))
            rewards.append(np.array([[x[2] / 10.0] for x in window], dtype=np.float32))
    return tuple(torch.tensor(np.stack(x), dtype=torch.float32) for x in (states, actions, next_states, rewards))


wm_train_s, wm_train_a, wm_train_ns, wm_train_r = episode_windows(train_episodes)
wm_val_s, wm_val_a, wm_val_ns, wm_val_r = episode_windows(val_episodes)


class TinyRecurrentWorldModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(3, 24), nn.SiLU())
        self.recurrent = nn.GRU(input_size=25, hidden_size=48, batch_first=True)
        self.state_head = nn.Linear(48, 3)
        self.reward_head = nn.Linear(48, 1)

    def forward(self, states, actions, hidden=None):
        latent = self.encoder(states)
        hidden_sequence, final_hidden = self.recurrent(torch.cat([latent, actions], dim=-1), hidden)
        return self.state_head(hidden_sequence), self.reward_head(hidden_sequence), final_hidden


seed_everything(1800)
world_model = TinyRecurrentWorldModel()
world_optimizer = torch.optim.AdamW(world_model.parameters(), lr=1.5e-3, weight_decay=1e-4)
generator = torch.Generator().manual_seed(1800)
for _ in range(700):
    ids = torch.randint(len(wm_train_s), (32,), generator=generator)
    predicted_states, predicted_rewards, _ = world_model(wm_train_s[ids], wm_train_a[ids])
    world_loss = F.mse_loss(predicted_states, wm_train_ns[ids]) + 0.25 * F.mse_loss(predicted_rewards, wm_train_r[ids])
    world_optimizer.zero_grad(); world_loss.backward(); world_optimizer.step()

with torch.no_grad():
    val_state_prediction, _, _ = world_model(wm_val_s, wm_val_a)
    teacher_forced_mse = F.mse_loss(val_state_prediction, wm_val_ns)
    imagined_state = wm_val_s[:, 0]
    hidden = None
    imagination_errors = []
    for time_index in range(wm_val_a.shape[1]):
        state_prediction, _, hidden = world_model(imagined_state[:, None, :], wm_val_a[:, time_index:time_index + 1], hidden)
        imagined_state = state_prediction[:, 0]
        imagination_errors.append(float(F.mse_loss(imagined_state, wm_val_ns[:, time_index])))

assert len(imagination_errors) == 20 and torch.isfinite(teacher_forced_mse)
print({"teacher-forced state MSE": round(float(teacher_forced_mse), 5), "imagination MSE": {"h=1": round(imagination_errors[0], 5), "h=10": round(imagination_errors[9], 5), "h=20": round(imagination_errors[19], 5)}})
```

</details>

这个确定性 GRU 是**机制模型**而不是 Dreamer 复现：Pendulum 完全可观测且维度很低，模型没有随机 posterior、图像 decoder、continuation model 或 imagined actor update。它的作用是揭示 teacher-forced prediction 与自主想象之间的差距。低 reconstruction error 并不意味着潜在状态保留了控制相关信息。


### **评估、可复现性与安全** {#evaluation-reproducibility-safety}

RL 评估对随机性格外敏感，因为数据收集、探索、初始化、replay 顺序与环境初始状态会相互作用。单个训练种子或最佳 checkpoint 不能证明算法稳健。[Deep Reinforcement Learning that Matters](https://arxiv.org/abs/1709.06560) 说明了实现和报告选择如何改变结论。

![可信的 RL 结果应拆分训练种子、固定评估起点、指标与不确定性报告。](assets/dl17-evaluation-protocol.svg){fig-align="center" width="78%" fig-alt="流程图展示多个训练种子在固定起点上评估，并报告回报、安全指标与不确定性区间。"}

可信协议应记录多个训练种子、固定测试起点、确定性与随机评估、相对于环境步数和墙钟时间的曲线、回报分布、终止约定、wrapper 与安全违例。奖励是工程化代理：智能体可能通过 specification gaming、不安全探索、模拟器伪影或模型漏洞来最大化它。

<details>
<summary><strong>Python：在固定起点上比较策略，并计算 bootstrap 区间与安全诊断</strong></summary>

```python
def evaluate_policy(policy, seeds, horizon=120):
    returns, saturation_rates, peak_speeds = [], [], []
    for seed in seeds:
        env = LocalPendulum(horizon=horizon)
        state = env.reset(int(seed))
        episode_return, actions, speeds = 0.0, [], []
        for _ in range(horizon):
            action = float(np.clip(policy(state), -2, 2))
            state, reward, _, truncated = env.step(action)
            episode_return += reward; actions.append(action); speeds.append(abs(float(state[2])))
            if truncated:
                break
        returns.append(episode_return)
        saturation_rates.append(np.mean(np.abs(actions) > 1.95))
        peak_speeds.append(max(speeds))
    return np.array(returns), float(np.mean(saturation_rates)), float(np.mean(peak_speeds))


def bootstrap_mean_interval(values, seed=1810, draws=2000):
    rng = np.random.default_rng(seed)
    samples = rng.choice(values, size=(draws, len(values)), replace=True).mean(1)
    return tuple(np.quantile(samples, [0.025, 0.975]))


evaluation_seeds = np.arange(18_100, 18_112)
policies = {"zero torque": lambda state: 0.0, "DQN": dqn_policy, "A2C": a2c_policy, "PPO": ppo_policy, "offline BC": bc_policy, "offline SAC objective": sac_policy}
evaluation = {}
for name, policy in policies.items():
    returns_array, saturation, peak_speed = evaluate_policy(policy, evaluation_seeds)
    low, high = bootstrap_mean_interval(returns_array, seed=1810 + len(name))
    evaluation[name] = {"mean return": round(float(returns_array.mean()), 1), "95% bootstrap interval": (round(float(low), 1), round(float(high), 1)), "torque saturation": round(saturation, 3), "mean peak speed": round(peak_speed, 2)}

assert all(np.isfinite(item["mean return"]) for item in evaluation.values())
assert all(item["95% bootstrap interval"][0] <= item["95% bootstrap interval"][1] for item in evaluation.values())
print(evaluation)
```

</details>

这里的区间量化固定初始状态之间的变化，而不是训练种子不确定性，因为每个策略都只训练了一次。这些数字比较的是一个小预算下的 Notebook 机制，不能用于普遍排列 DQN、PPO、SAC 或 BC。Benchmark 结论需要独立训练种子、经过调优的 baseline、跨运行置信区间与公认实现。


### **章节对比与总结** {#chapter-comparison-summary}

深度 RL 方法的关键区别在于：**学习什么对象**、**数据从哪里产生**，以及**是否使用环境模型**。

| 方法 | 学习对象 | 数据机制 | 动作空间 | 主要优势 | 典型风险 |
|---|---|---|---|---|---|
| DQN | 动作价值函数 | 在线、off-policy replay | 离散 | 复用经验 | 自举不稳定与高估 |
| REINFORCE | 随机策略 | 在线、on-policy | 均可 | 直接且简单 | 梯度方差高 |
| Actor-Critic | 策略与 critic | 通常在线 | 均可 | 更低方差的更新 | critic 偏差或滞后 |
| PPO | 裁剪的 actor-critic 目标 | 在线、on-policy | 均可 | 稳健的实用 baseline | 需要新 rollout；裁剪并非保证 |
| SAC | 熵正则 actor 与双 critic | 在线、off-policy replay | 连续 | replay 复用与探索 | critic 支持范围和实现敏感性 |
| Behavioral cloning | 策略 | 固定 demonstration | 均可 | 稳定监督目标 | 累积 covariate shift |
| 离线 RL / CQL | 保守策略与价值 | 固定日志数据 | 均可 | 不需要新交互 | 数据支持与选择泄漏 |
| 基于模型的 RL | 动力学/奖励加 planner 或策略 | 在线或离线 | 均可 | 样本高效的模拟 | 模型偏差累积 |
| 世界模型智能体 | 潜在动力学、奖励、actor、critic | 真实与想象轨迹 | 均可 | 在想象中学习策略 | 利用潜在模型误差 |

一个实用的选择顺序是：

1. 选择算法前先定义状态、动作、奖励、时域、终止、约束与评估方式。
2. 对有意义的有限动作使用 DQN；把 PPO 作为清晰的 on-policy baseline；连续控制且需要 replay 效率时考虑 SAC。
3. 有 demonstration 时使用模仿学习，但要审查部署时的 covariate shift。
4. 把静态日志数据视为离线 RL，并明确测量动作支持范围。
5. 当交互成本足以抵消模型偏差管理成本时，再引入学习动力学。
6. 使用潜在世界模型时同时检查开环预测、不确定性与真实环境验证。
7. 在平均回报之外，报告跨种子和起点的分布、学习成本、安全指标与失败案例。

本章在使用 RL 进行偏好对齐之前结束。下一章会再次使用 PPO 作为 RLHF 的一个组件；此时 prompt、生成回复、reward model、KL 约束和人类偏好数据都会改变目标及其失败模式。
